# P1 SGN clustering

Set the paths marked in the code before running this notebook.
Reproduces the selected P1 Type Ia, Type Ib, Type Ic, and Type II UMAP and marker DotPlot from `P1_2.ipynb`. The final cell verifies or writes one raw-count H5AD for later integration.


In [ ]:
#1. =========== Load P1 counts and set the random seed
# Replace each ROOT placeholder with the required file or directory path before running.
import csv
import gzip
import hashlib
import logging
import os
import random
import re
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path

for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[key] = "1"

import anndata as ad
import harmonypy as hm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from scipy import sparse
from threadpoolctl import threadpool_limits

RANDOM_SEED, N_THREADS = 42, 1
ROOT = Path(".")  # Set the project root before running this template.
if ROOT is None:
    raise FileNotFoundError("Project input counts were not found.")
STAGE, DATASET_ID = "P1", "GSE195500"
# Set the original count-data file or directory path.
SOURCE = ROOT
LIBRARIES = {"13": ("GSM5839485", "P1_1"), "14": ("GSM5839486", "P1_2")}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
thread_guard = threadpool_limits(limits=N_THREADS)
sc.settings.n_jobs, sc.settings.verbosity = N_THREADS, 0
logging.getLogger("harmonypy").setLevel(logging.WARNING)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 10,
    "figure.dpi": 120, "figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})

with gzip.open(SOURCE, "rt") as handle:
    reader = csv.reader(handle)
    source_ids, first_row = pd.Index(next(reader)), next(reader)
assert len(source_ids) == 5441 and len(first_row) == 5442 and source_ids.is_unique
assert all(re.fullmatch(r"[ACGT]{16}-1_\d+", x) for x in source_ids)
p1_ids = source_ids[[x.rsplit("_", 1)[1] in LIBRARIES for x in source_ids]]
counts = pd.read_csv(SOURCE, header=None, skiprows=1, names=["gene", *source_ids], usecols=["gene", *p1_ids], index_col="gene", keep_default_na=False)
assert counts.shape[0] == 17999 and counts.index.is_unique and counts.columns.equals(p1_ids)
values = counts.to_numpy()
assert np.isfinite(values).all() and (values >= 0).all() and (values == np.floor(values)).all()
assert values.max() <= np.iinfo(np.int32).max

var = pd.DataFrame(index=counts.index.astype(str))
var.index.name = "gene_symbol"
var["mt"] = var.index.str.match(r"(?i)^mt[-_]")
var["hb"] = var.index.str.match(r"(?i)^hb[ab]-")
var["ribo"] = var.index.str.match(r"^Rp[sl]\d")
var["technical"] = var.index.str.match(r"(?i)^ercc[-_]\d+$|^(e?gfp|tdtomato|mcherry|wpre)$")
raw_by_library = {}
for suffix, (gsm, sample) in LIBRARIES.items():
    library = f"{DATASET_ID}_L{suffix}"
    ids = counts.columns[counts.columns.str.endswith(f"_{suffix}")]
    X = sparse.csr_matrix(counts.loc[:, ids].to_numpy().T, dtype=np.int32)
    obs = pd.DataFrame(index=pd.Index([f"{DATASET_ID}__{library}__{x}" for x in ids], name="cell_id_global"))
    obs["original_cell_id"], obs["barcode"] = ids.to_numpy(), [x.rsplit("_", 1)[0] for x in ids]
    obs["dataset_id"], obs["library_id"], obs["GSM"] = DATASET_ID, library, gsm
    obs["sample_id"] = obs["biological_replicate_id"] = f"{DATASET_ID}_{sample}"
    obs["stage_original"] = obs["stage_standardized"] = STAGE
    obs["technology"], obs["cochlear_region"], obs["genotype"] = "10x_3prime_v3.1", "whole_ganglion", "WT"
    obs["source_file"], obs["author_preselected"] = str(SOURCE), True
    obj = ad.AnnData(X=X, obs=obs, var=var.copy())
    obj.layers["counts"] = X.copy()
    raw_by_library[library] = obj


In [ ]:
#2. =========== P1 and E18.5 quality control
SCRUBLET_RATE, SCRUBLET_PCS = .05, 20
DOUBLET_REVIEW_QUANTILE = .95
NEURON_MARKERS = ["Snap25", "Tubb3", "Stmn2", "Elavl3", "Rbfox3"]
EXCLUSION = {
    "Immune": ["Ptprc", "Lyz2", "Tyrobp", "C1qa"],
    "Glia": ["Mpz", "Pmp22", "Mag", "Mbp", "Plp1", "Dhh", "Sox10"],
    "Hair": ["Myo7a", "Pou4f3", "Otof"],
    "Epithelial": ["Epcam", "Krt8", "Krt18"],
    "Vascular": ["Pecam1", "Kdr", "Esam"],
    "Mesenchymal": ["Col1a1", "Col1a2", "Dcn"],
}
QC_PANELS = {**EXCLUSION, "Erythroid": ["Gypa", "Klf1", "Alas2", "Slc4a1"]}
source_genes = next(iter(raw_by_library.values())).var_names
panel_coverage = {name: [g for g in genes if g in source_genes]
                  for name, genes in QC_PANELS.items()}

for library, a in raw_by_library.items():
    X = a.layers["counts"]
    total = np.asarray(X.sum(axis=1)).ravel()
    a.obs["total_counts"] = total
    a.obs["n_genes"] = X.getnnz(axis=1)
    a.obs["pct_hb"] = 100 * np.asarray(X[:, a.var["hb"].to_numpy()].sum(axis=1)).ravel() / np.maximum(total, 1)
    a.obs["pct_ribo"] = 100 * np.asarray(X[:, a.var["ribo"].to_numpy()].sum(axis=1)).ravel() / np.maximum(total, 1)
    a.obs["neuronal_genes_detected"] = X[:, a.var_names.isin(NEURON_MARKERS)].getnnz(axis=1)
    for name, available in panel_coverage.items():
        detected = X[:, a.var_names.isin(available)].getnnz(axis=1)
        a.obs[f"{name.lower()}_genes_detected"] = detected
        a.obs[f"review_{name.lower()}"] = (
            detected >= 2 if len(available) >= 2
            else pd.Series(pd.NA, index=a.obs_names, dtype="boolean")
        )
    a.obs["blood_review"] = (
        a.obs["review_immune"].fillna(False).astype(bool)
        | a.obs["review_erythroid"].fillna(False).astype(bool)
    )
    a.obs["non_target_review"] = pd.concat(
        [a.obs[f"review_{name.lower()}"].fillna(False).astype(bool)
         for name in QC_PANELS], axis=1
    ).any(axis=1)
    probe = a[:, ~(a.var["hb"] | a.var["technical"]).to_numpy()].copy()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide",
                                category=RuntimeWarning, module=r"scanpy\.preprocessing\._scrublet\._core")
        sc.pp.scrublet(probe, expected_doublet_rate=SCRUBLET_RATE,
                      n_prin_comps=SCRUBLET_PCS, use_approx_neighbors=False,
                      random_state=RANDOM_SEED, threshold=1, verbose=False)
    scores = probe.obs["doublet_score"].to_numpy()
    a.obs["doublet_score"] = scores
    a.obs["doublet_candidate"] = scores >= np.quantile(scores, DOUBLET_REVIEW_QUANTILE)

qc_by_library = {}
for library, a in raw_by_library.items():
    zero_content = a.obs["total_counts"].eq(0) | a.obs["n_genes"].eq(0)
    a.obs["qc_pass"] = ~zero_content
    a.obs["exclusion_reason"] = np.where(zero_content, "zero_counts_or_genes", "")
    a.obs["qc_review_flag"] = a.obs["non_target_review"] | a.obs["doublet_candidate"]
    qc_by_library[library] = a[(~zero_content).to_numpy()].copy()
qc_full = ad.concat(list(qc_by_library.values()), join="inner", merge="same", index_unique=None)
assert qc_full.obs_names.is_unique and qc_full.var_names.equals(source_genes)
assert (qc_full.X != qc_full.layers["counts"]).nnz == 0

E18_REPLICATES = {"10": "E18_R1", "11": "E18_R2", "12": "E18_R3"}
e18_ids = source_ids[[x.rsplit("_", 1)[1] in E18_REPLICATES for x in source_ids]]
e18_counts = pd.read_csv(SOURCE, header=None, skiprows=1, names=["gene", *source_ids],
                         usecols=["gene", *e18_ids], index_col="gene", keep_default_na=False)
assert len(e18_ids) == 862 and e18_counts.index.equals(source_genes)
e18_values = e18_counts.to_numpy()
assert np.isfinite(e18_values).all() and (e18_values >= 0).all()
assert (e18_values == np.floor(e18_values)).all()

e18_by_library = {}
for suffix, replicate in E18_REPLICATES.items():
    library = f"{DATASET_ID}_L{suffix}"
    ids = e18_counts.columns[e18_counts.columns.str.endswith(f"_{suffix}")]
    X = sparse.csr_matrix(e18_counts.loc[:, ids].to_numpy().T, dtype=np.int32)
    obs = pd.DataFrame(index=pd.Index(
        [f"{DATASET_ID}__{library}__{x}" for x in ids], name="cell_id_global"))
    obs["original_cell_id"] = ids.to_numpy()
    obs["barcode"] = [x.rsplit("_", 1)[0] for x in ids]
    obs["dataset_id"], obs["library_id"], obs["GSM"] = DATASET_ID, library, pd.NA
    obs["sample_id"] = obs["biological_replicate_id"] = f"{DATASET_ID}_{replicate}"
    obs["stage_original"], obs["stage_standardized"] = "E18", "E18.5"
    obs["technology"], obs["cochlear_region"], obs["genotype"] = "10x_3prime_v3.1", "whole_ganglion", "WT"
    obs["source_file"], obs["author_preselected"] = str(SOURCE), True

    a = ad.AnnData(X=X, obs=obs, var=var.copy())
    a.layers["counts"] = X.copy()
    total = np.asarray(X.sum(axis=1)).ravel()
    a.obs["total_counts"], a.obs["n_genes"] = total, X.getnnz(axis=1)
    a.obs["pct_hb"] = 100 * np.asarray(X[:, a.var["hb"].to_numpy()].sum(axis=1)).ravel() / np.maximum(total, 1)
    a.obs["pct_ribo"] = 100 * np.asarray(X[:, a.var["ribo"].to_numpy()].sum(axis=1)).ravel() / np.maximum(total, 1)
    a.obs["neuronal_genes_detected"] = X[:, a.var_names.isin(NEURON_MARKERS)].getnnz(axis=1)

    for name, genes in panel_coverage.items():
        detected = X[:, a.var_names.isin(genes)].getnnz(axis=1)
        a.obs[f"{name.lower()}_genes_detected"] = detected
        a.obs[f"review_{name.lower()}"] = detected >= 2 if len(genes) >= 2 else False
    a.obs["blood_review"] = a.obs["review_immune"] | a.obs["review_erythroid"]
    a.obs["non_target_review"] = a.obs[[f"review_{name.lower()}" for name in QC_PANELS]].any(axis=1)

    e18_by_library[library] = a

del e18_counts, e18_values
qc_preview = pd.concat([a.obs for a in e18_by_library.values()] + [qc_full.obs])
library_order = list(e18_by_library) + list(qc_by_library)
fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.5))
for ax, metric, title in zip(axes[:2], ("total_counts", "n_genes"), ("UMI counts", "Detected genes")):
    values = [np.log10(1 + qc_preview.loc[qc_preview["library_id"].eq(lib), metric].to_numpy())
              for lib in library_order]
    ax.violinplot(values, showmedians=True, showextrema=False)
    ax.set_xticks(range(1, len(library_order) + 1), [lib.rsplit("_", 1)[-1] for lib in library_order])
    ax.set(xlabel="Library", ylabel=f"log10(1 + {title})", title=title)

for stage, color in (("E18.5", "#557B9F"), ("P1", "#B7864A")):
    rows = qc_preview["stage_standardized"].eq(stage)
    axes[2].scatter(np.log10(1 + qc_preview.loc[rows, "total_counts"]),
                    np.log10(1 + qc_preview.loc[rows, "n_genes"]),
                    s=7, alpha=.55, linewidths=0, color=color, label=stage)
flagged = qc_preview["non_target_review"].to_numpy(dtype=bool)
axes[2].scatter(np.log10(1 + qc_preview.loc[flagged, "total_counts"]),
                np.log10(1 + qc_preview.loc[flagged, "n_genes"]),
                s=23, marker="x", linewidths=.8, color="#B22222", label="Non-target review")
axes[2].set(xlabel="log10(1 + UMI)", ylabel="log10(1 + genes)", title="Complexity and review flags")
axes[2].legend(frameon=False, fontsize=7)
fig.tight_layout()
plt.show()


In [ ]:
#3. =========== Joint E18.5/P1 expression and P1 reference embedding
ep_raw = ad.concat([*e18_by_library.values(), qc_full], join="inner", merge="same", index_unique=None)
usable = ((ep_raw.layers["counts"].getnnz(axis=0) >= 3)
          & ~(ep_raw.var["mt"] | ep_raw.var["hb"] | ep_raw.var["technical"]).to_numpy())
ep_expression = ep_raw[:, usable].copy()
ep_expression.X = ep_expression.layers["counts"].astype(np.float32).copy()
sc.pp.normalize_total(ep_expression, target_sum=10000)
sc.pp.log1p(ep_expression)

HVG_N, N_PCS, N_NEIGHBORS = 3000, 25, 15
RESOLUTION, HARMONY_THETA, UMAP_MIN_DIST = 2.0, 2, 0.35
tune_full = ep_expression.copy()
sc.pp.highly_variable_genes(tune_full, n_top_genes=HVG_N, flavor="seurat_v3",
                            layer="counts", batch_key="library_id", span=0.3)
hvg = tune_full[:, tune_full.var["highly_variable"]].copy()
hvg.X = hvg.X.toarray() if sparse.issparse(hvg.X) else np.asarray(hvg.X)
sc.pp.scale(hvg, max_value=10)
sc.tl.pca(hvg, n_comps=N_PCS, svd_solver="arpack", random_state=RANDOM_SEED)
corrected = hm.run_harmony(
    np.ascontiguousarray(hvg.obsm["X_pca"], dtype=np.float32),
    tune_full.obs, ["library_id"], theta=HARMONY_THETA,
    max_iter_harmony=20, nclust=10, random_state=RANDOM_SEED, verbose=False
).Z_corr
tune_full.obsm["X_pca"] = corrected if corrected.shape == (tune_full.n_obs, N_PCS) else corrected.T
tune_full.obsm["X_pca_harmony"] = tune_full.obsm["X_pca"].copy()
sc.pp.neighbors(tune_full, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS,
                transformer="sklearn", random_state=RANDOM_SEED)
sc.tl.leiden(tune_full, resolution=RESOLUTION, flavor="igraph",
             n_iterations=2, directed=False, random_state=RANDOM_SEED,
             key_added="cluster_tune")
sc.tl.umap(tune_full, min_dist=UMAP_MIN_DIST, random_state=RANDOM_SEED)
p1_tune = tune_full[tune_full.obs["stage_standardized"].eq("P1").to_numpy()].copy()


In [ ]:
#4. =========== Fix Type II and resolve Type I clusters
TYPEII_CLUSTER = "13"          # Reviewed cluster from #8 at resolution 2.0
EXPECTED_P1_TYPEII = 18
TYPEI_HVG = 2000
TYPEI_PCS = 25
TYPEI_NEIGHBORS = 15
TYPEI_RESOLUTION = 1.2
TYPEI_HARMONY_THETA = 2


if not tune_full.obs_names.equals(ep_expression.obs_names):
    raise ValueError("The selected clustering does not match ep_expression cell order.")

type_ii_ids = tune_full.obs_names[
    tune_full.obs["cluster_tune"].astype(str).eq(TYPEII_CLUSTER)
]
ep_type_ii_fixed = ep_raw[ep_raw.obs_names.isin(type_ii_ids)].copy()
p1_type_ii_fixed = ep_type_ii_fixed[
    ep_type_ii_fixed.obs["stage_standardized"].eq("P1").to_numpy()
].copy()
if p1_type_ii_fixed.n_obs != EXPECTED_P1_TYPEII:
    raise ValueError("Cluster 13 no longer matches the reviewed #8 result; check the selected parameters.")
p1_type_ii_fixed.obs["cell_type"] = "Type II"

ep_type_i_expression = ep_expression[
    ~ep_expression.obs_names.isin(type_ii_ids)
].copy()
gene_keep = np.asarray(
    ep_type_i_expression.layers["counts"].getnnz(axis=0)
).ravel() >= 3
ep_type_i_expression = ep_type_i_expression[:, gene_keep].copy()
a = ep_type_i_expression.copy()
sc.pp.highly_variable_genes(
    a, n_top_genes=TYPEI_HVG, flavor="seurat_v3",
    layer="counts", batch_key="library_id", span=0.3
)
h = a[:, a.var["highly_variable"]].copy()
h.X = h.X.toarray() if sparse.issparse(h.X) else np.asarray(h.X)
sc.pp.scale(h, max_value=10)
sc.tl.pca(h, n_comps=TYPEI_PCS, svd_solver="arpack",
          random_state=RANDOM_SEED)

corrected = hm.run_harmony(
    np.ascontiguousarray(h.obsm["X_pca"], dtype=np.float32),
    a.obs, ["library_id"], theta=TYPEI_HARMONY_THETA,
    max_iter_harmony=20, nclust=10,
    random_state=RANDOM_SEED, verbose=False
).Z_corr
a.obsm["X_pca"] = corrected if corrected.shape == (a.n_obs, TYPEI_PCS) else corrected.T
a.obsm["X_pca_harmony"] = a.obsm["X_pca"].copy()

sc.pp.neighbors(a, n_neighbors=TYPEI_NEIGHBORS, n_pcs=TYPEI_PCS,
                transformer="sklearn", random_state=RANDOM_SEED)
sc.tl.leiden(a, resolution=TYPEI_RESOLUTION, flavor="igraph",
             n_iterations=2, directed=False, random_state=RANDOM_SEED,
             key_added="cluster_type_i")
ep_type_i_clustered = a

p1 = a[a.obs["stage_standardized"].eq("P1").to_numpy()].copy()
p1.obs["cluster_type_i"] = p1.obs["cluster_type_i"].cat.remove_unused_categories()
order = sorted(p1.obs["cluster_type_i"].cat.categories.tolist(), key=int)
p1.obs["cluster_type_i"] = pd.Categorical(
    p1.obs["cluster_type_i"].astype(str), categories=order, ordered=True
)
p1_type_i_clustered = p1


In [ ]:
#5. =========== Recluster unresolved Type I cells
FIXED_CLUSTER_TYPES = {
    "0": "Type Ia", "2": "Type Ia", "7": "Type Ia",
    "4": "Type Ib", "6": "Type Ic"
}
FOCUS_HVG = 2000
FOCUS_PCS, FOCUS_NEIGHBORS, FOCUS_RESOLUTION = 25, 15, 0.8
FOCUS_HARMONY_THETA = 2

p1_base = p1_type_i_clustered
joint_base = ep_type_i_clustered
if not joint_base.obs_names.equals(ep_type_i_expression.obs_names):
    raise ValueError("The HVG2000 clustering does not match the Type I expression object.")

p1_cluster = p1_base.obs["cluster_type_i"].astype(str)
unresolved_clusters = sorted(set(p1_cluster) - set(FIXED_CLUSTER_TYPES), key=int)
unresolved_ids = p1_base.obs_names[p1_cluster.isin(unresolved_clusters)]

fixed_labels = p1_cluster.map(FIXED_CLUSTER_TYPES).dropna()
fixed_labels = pd.concat([
    fixed_labels,
    pd.Series("Type II", index=p1_type_ii_fixed.obs_names)
])
p1_fixed = qc_full[qc_full.obs_names.isin(fixed_labels.index)].copy()
p1_fixed.obs["cell_type"] = fixed_labels.reindex(p1_fixed.obs_names).to_numpy()
p1_unresolved_raw = qc_full[qc_full.obs_names.isin(unresolved_ids)].copy()

if (p1_fixed.n_obs, p1_unresolved_raw.n_obs) != (243, 221):
    raise ValueError("Fixed/unresolved P1 cell counts changed; review #8–#10 before continuing.")

joint_unresolved = joint_base.obs["cluster_type_i"].astype(str).isin(unresolved_clusters).to_numpy()
ep_unresolved_expression = ep_type_i_expression[joint_unresolved].copy()
gene_keep = np.asarray(ep_unresolved_expression.layers["counts"].getnnz(axis=0)).ravel() >= 3
ep_unresolved_expression = ep_unresolved_expression[:, gene_keep].copy()
a = ep_unresolved_expression.copy()
sc.pp.highly_variable_genes(
    a, n_top_genes=FOCUS_HVG, flavor="seurat",
    batch_key="library_id"
)
h = a[:, a.var["highly_variable"]].copy()
h.X = h.X.toarray() if sparse.issparse(h.X) else np.asarray(h.X)
sc.pp.scale(h, max_value=10)
sc.tl.pca(h, n_comps=FOCUS_PCS, svd_solver="arpack", random_state=RANDOM_SEED)

corrected = hm.run_harmony(
    np.ascontiguousarray(h.obsm["X_pca"], dtype=np.float32),
    a.obs, ["library_id"], theta=FOCUS_HARMONY_THETA,
    max_iter_harmony=20, nclust=10, random_state=RANDOM_SEED, verbose=False
).Z_corr
a.obsm["X_pca"] = corrected if corrected.shape == (a.n_obs, FOCUS_PCS) else corrected.T
a.obsm["X_pca_harmony"] = a.obsm["X_pca"].copy()

sc.pp.neighbors(a, n_neighbors=FOCUS_NEIGHBORS, n_pcs=FOCUS_PCS,
                transformer="sklearn", random_state=RANDOM_SEED)
sc.tl.leiden(a, resolution=FOCUS_RESOLUTION, flavor="igraph",
             n_iterations=2, directed=False, random_state=RANDOM_SEED,
             key_added="cluster_unresolved")
ep_unresolved_clustered = a

p1 = a[a.obs["stage_standardized"].eq("P1").to_numpy()].copy()
p1.obs["cluster_unresolved"] = p1.obs["cluster_unresolved"].cat.remove_unused_categories()
order = sorted(p1.obs["cluster_unresolved"].cat.categories.tolist(), key=int)
p1.obs["cluster_unresolved"] = pd.Categorical(
    p1.obs["cluster_unresolved"].astype(str), categories=order, ordered=True
)
p1_unresolved_clustered = p1


In [ ]:
#6. =========== Add Type Ib and Type Ic cells
NEW_IB_CLUSTER = "1"
# Add the reviewed Type Ib cluster.
focus_p1 = p1_unresolved_clustered
new_ib_ids = focus_p1.obs_names[
    focus_p1.obs["cluster_unresolved"].astype(str).eq(NEW_IB_CLUSTER).to_numpy()
]
fixed_labels_round2 = pd.concat([
    p1_fixed.obs["cell_type"].astype(str),
    pd.Series("Type Ib", index=new_ib_ids, name="cell_type"),
])
p1_fixed_round2 = qc_full[qc_full.obs_names.isin(fixed_labels_round2.index)].copy()
p1_fixed_round2.obs["cell_type"] = fixed_labels_round2.reindex(p1_fixed_round2.obs_names).to_numpy()
p1_remaining_round2 = p1_unresolved_raw[
    ~p1_unresolved_raw.obs_names.isin(new_ib_ids)
].copy()

if (p1_fixed_round2.n_obs, p1_remaining_round2.n_obs) != (298, 166):
    raise ValueError("P1 cell counts changed; check the selected clusters.")

HVG_N = 2000
N_PCS = 25
N_NEIGHBORS = 15
RESOLUTION = 1.2
HARMONY_THETA = 2
REMOVED_IB_CLUSTER = "1"

joint_previous = ep_unresolved_clustered
if not joint_previous.obs_names.equals(ep_unresolved_expression.obs_names):
    raise ValueError("The previous clustering and expression objects have different cell orders.")

keep_joint = ~joint_previous.obs["cluster_unresolved"].astype(str).eq(
    REMOVED_IB_CLUSTER
).to_numpy()
ep_remaining_expression = ep_unresolved_expression[keep_joint].copy()
gene_keep = np.asarray(
    ep_remaining_expression.layers["counts"].getnnz(axis=0)
).ravel() >= 3
ep_remaining_expression = ep_remaining_expression[:, gene_keep].copy()

ep_remaining_round3 = ep_remaining_expression.copy()
sc.pp.highly_variable_genes(
    ep_remaining_round3, n_top_genes=HVG_N,
    flavor="seurat", batch_key="library_id"
)
hvg = ep_remaining_round3[:, ep_remaining_round3.var["highly_variable"]].copy()
hvg.X = hvg.X.toarray() if sparse.issparse(hvg.X) else np.asarray(hvg.X)
sc.pp.scale(hvg, max_value=10)
sc.tl.pca(hvg, n_comps=N_PCS, svd_solver="arpack", random_state=RANDOM_SEED)

corrected = hm.run_harmony(
    np.ascontiguousarray(hvg.obsm["X_pca"], dtype=np.float32),
    ep_remaining_round3.obs, ["library_id"], theta=HARMONY_THETA,
    max_iter_harmony=20, nclust=10, random_state=RANDOM_SEED, verbose=False
).Z_corr
ep_remaining_round3.obsm["X_pca"] = (
    corrected if corrected.shape == (ep_remaining_round3.n_obs, N_PCS)
    else corrected.T
)
ep_remaining_round3.obsm["X_pca_harmony"] = ep_remaining_round3.obsm["X_pca"].copy()

sc.pp.neighbors(
    ep_remaining_round3, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS,
    transformer="sklearn", random_state=RANDOM_SEED
)
sc.tl.leiden(
    ep_remaining_round3, resolution=RESOLUTION, flavor="igraph",
    n_iterations=2, directed=False, random_state=RANDOM_SEED,
    key_added="cluster_remaining"
)

p1_remaining_round3 = ep_remaining_round3[
    ep_remaining_round3.obs["stage_standardized"].eq("P1").to_numpy()
].copy()
if p1_remaining_round3.n_obs != p1_remaining_round2.n_obs:
    raise ValueError("The remaining P1 cell count changed unexpectedly.")

p1_remaining_round3.obs["cluster_remaining"] = (
    p1_remaining_round3.obs["cluster_remaining"].cat.remove_unused_categories()
)
cluster_order = sorted(
    p1_remaining_round3.obs["cluster_remaining"].cat.categories.tolist(),
    key=int
)
p1_remaining_round3.obs["cluster_remaining"] = pd.Categorical(
    p1_remaining_round3.obs["cluster_remaining"].astype(str),
    categories=cluster_order, ordered=True
)

FIX_IC_CLUSTER = "7"
if not ep_remaining_round3.obs_names.equals(ep_remaining_expression.obs_names):
    raise ValueError("Round-3 clusters and expression object have different cell orders.")

new_ic_ids = p1_remaining_round3.obs_names[
    p1_remaining_round3.obs["cluster_remaining"].astype(str).eq(FIX_IC_CLUSTER).to_numpy()
]
if len(new_ic_ids) != 16:
    raise ValueError("Reviewed Type Ic cluster 7 no longer contains 16 P1 cells.")

fixed_labels = pd.concat([
    p1_fixed_round2.obs["cell_type"].astype(str),
    pd.Series("Type Ic", index=new_ic_ids, name="cell_type"),
])
p1_fixed_round3 = qc_full[qc_full.obs_names.isin(fixed_labels.index)].copy()
p1_fixed_round3.obs["cell_type"] = fixed_labels.reindex(p1_fixed_round3.obs_names).to_numpy()


In [ ]:
#7. =========== Final four-type UMAP
TYPE_ORDER = ["Type Ia", "Type Ib", "Type Ic", "Type II"]
TYPE_COLORS = {
    "Type Ia": "#4E79A7", "Type Ib": "#E15759",
    "Type Ic": "#59A14F", "Type II": "#B07AA1",
}
final_view = ep_expression[p1_fixed_round3.obs_names, :].copy()
final_view.obs["cell_type"] = pd.Categorical(
    p1_fixed_round3.obs["cell_type"].reindex(final_view.obs_names).astype(str),
    categories=TYPE_ORDER, ordered=True,
)
assert final_view.obs["cell_type"].value_counts().reindex(TYPE_ORDER).tolist() == [145, 114, 37, 18]
final_view.obsm["X_umap"] = np.asarray(p1_tune[final_view.obs_names, :].obsm["X_umap"]).copy()

xy = np.asarray(final_view.obsm["X_umap"])
center = (xy.min(axis=0) + xy.max(axis=0)) / 2
span = 1.08 * np.max(xy.max(axis=0) - xy.min(axis=0))
xlim = (center[0] - span / 2, center[0] + span / 2)
ylim = (center[1] - span / 2, center[1] + span / 2)
labels = final_view.obs["cell_type"].astype(str).to_numpy()
fig, ax = plt.subplots(figsize=(7.1, 5.3))
fig.subplots_adjust(left=0.12, right=0.65, bottom=0.13, top=0.89)
for cell_type in TYPE_ORDER:
    mask = labels == cell_type
    ax.scatter(xy[mask, 0], xy[mask, 1], s=18,
               color=TYPE_COLORS[cell_type], alpha=0.82, linewidths=0)
handles = [Line2D([0], [0], marker="o", linestyle="None",
                  markerfacecolor=TYPE_COLORS[t], markeredgecolor="none",
                  markersize=5.5) for t in TYPE_ORDER]
legend_labels = [f"{t} (n={np.sum(labels == t)})" for t in TYPE_ORDER]
ax.legend(handles, legend_labels, loc="upper left", bbox_to_anchor=(1.02, 1),
          frameon=False, fontsize=8, handletextpad=0.35)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_box_aspect(1)
ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("P1 four types")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
#8. =========== Final P1 marker DotPlot
dot_cmap = LinearSegmentedColormap.from_list(
    "sgn_dot", ["#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F"]
)
MAX_DOT_AREA = 250
COLUMN_GAP_RATIO = 0.5
workbook_genes = [
    "Calb2", "Lypd1", "Calb1", "Id1", "Tle4", "Runx1", "Ntng1",
    "Pou4f1", "Espn", "Efna5", "Tac1", "Th", "Prph"
]
candidate_panel = {
    "Type Ia": ["Calb2", "Id1", "Tle4", "Fgf10", "Shox2"],
    "Type Ib": ["Calb1", "Runx1", "Ntng1"],
    "Type Ic": ["Lypd1", "Pou4f1", "Espn", "Mgat4c", "Bhlhe22",
                "Tle2", "Runx1t1", "Tgfb2", "Igfbp2"],
    "Type II": ["Efna5", "Tac1", "Th", "Prph",
                "Plk5", "Calca", "Sox9", "Tshz3"],
}
available = set(final_view.var_names)
marker_panel = {cell_type: [gene for gene in candidates if gene in available]
                for cell_type, candidates in candidate_panel.items()}
genes = [gene for cell_type in TYPE_ORDER for gene in marker_panel[cell_type]]

expression = final_view[:, genes].X
counts = final_view[:, genes].layers["counts"]
expression = expression.toarray() if sparse.issparse(expression) else np.asarray(expression)
counts = counts.toarray() if sparse.issparse(counts) else np.asarray(counts)
mean_final = pd.DataFrame(expression, columns=genes).groupby(labels, sort=False).mean().reindex(TYPE_ORDER)
fractions = pd.DataFrame(counts > 0, columns=genes).groupby(labels, sort=False).mean().reindex(TYPE_ORDER)

# Use the original color-scale reference without creating another plot.
reference_view = ep_expression[p1_fixed.obs_names, genes]
reference_expression = reference_view.X
reference_expression = (reference_expression.toarray() if sparse.issparse(reference_expression)
                        else np.asarray(reference_expression))
reference_labels = p1_fixed.obs["cell_type"].reindex(reference_view.obs_names).astype(str).to_numpy()
mean_reference = (pd.DataFrame(reference_expression, columns=genes)
                  .groupby(reference_labels, sort=False).mean().reindex(TYPE_ORDER))
combined_means = pd.concat([mean_final, mean_reference], ignore_index=True)
gene_center = combined_means.mean(axis=0)
gene_sd = combined_means.std(axis=0, ddof=0).replace(0, 1)
scaled_mean = mean_final.sub(gene_center, axis=1).div(gene_sd, axis=1)

dot_diameter_pt = 2 * np.sqrt(MAX_DOT_AREA / np.pi)
dot_width = max(10, len(genes) * (1 + COLUMN_GAP_RATIO) * dot_diameter_pt / 72 / 0.76)
norm = Normalize(vmin=-1, vmax=1, clip=True)
xx, yy = np.meshgrid(np.arange(len(genes)), np.arange(len(TYPE_ORDER))[::-1])
fig, ax = plt.subplots(figsize=(dot_width, 5.1))
fig.subplots_adjust(left=0.17, right=0.93, bottom=0.25, top=0.63)
pct = fractions.to_numpy()
color = scaled_mean.to_numpy()
visible = pct.ravel() > 0
ax.scatter(xx.ravel()[visible], yy.ravel()[visible],
           s=MAX_DOT_AREA * pct.ravel()[visible],
           c=color.ravel()[visible], cmap=dot_cmap, norm=norm, edgecolors="none")

start = 0
for cell_type in TYPE_ORDER:
    count = len(marker_panel[cell_type])
    ax.plot([start - 0.42, start + count - 0.58], [3.72, 3.72],
            color=TYPE_COLORS[cell_type], linewidth=3, solid_capstyle="butt")
    ax.text(start + (count - 1) / 2, 3.91, cell_type,
            ha="center", va="bottom", fontsize=8)
    start += count
gene_labels = [gene if gene in workbook_genes else f"{gene}*" for gene in genes]
ax.set_xticks(np.arange(len(genes)))
ax.set_xticklabels(gene_labels, rotation=55, ha="right", fontsize=8)
ax.set_yticks(np.arange(len(TYPE_ORDER))[::-1])
ax.set_yticklabels(TYPE_ORDER, fontsize=9)
ax.set_xlim(-0.5, len(genes) - 0.5)
ax.set_ylim(-0.5, 4.4)
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
cax = fig.add_axes([0.17, 0.82, 0.31, 0.026])
colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=dot_cmap), cax=cax,
                        orientation="horizontal", ticks=[-1, -0.5, 0, 0.5, 1])
colorbar.outline.set_visible(False)
colorbar.ax.tick_params(length=0, labelsize=7)
cax.set_title("Avg. Exp. (gene-wise z)", fontsize=8, loc="left", pad=6)
levels = [0, 0.25, 0.5, 0.75, 1]
handles = [Line2D([0], [0], marker="o", linestyle="None",
                  markerfacecolor="#777777", markeredgecolor="none",
                  markersize=dot_diameter_pt * np.sqrt(value)) for value in levels]
fig.legend(handles, ["0", "25", "50", "75", "100"], title="% Exp.", ncol=5,
           loc="upper right", bbox_to_anchor=(0.94, 0.91), frameon=False,
           fontsize=7, title_fontsize=8, handletextpad=0.2, columnspacing=0.7)
fig.text(0.17, 0.96, "P1 four types", fontsize=10)
fig.text(0.17, 0.025, "* Literature supplement; unmarked genes are from the P1 workbook.", fontsize=7)
plt.show()


In [ ]:
#9. =========== Canonical raw-count H5AD for integration
# Set the result file path; uses P1_GSE195500__L13-L14__SGN_for_integration.h5ad generated by P1 clustering export.
out_path = (
    ROOT
)
# Set the data file path; uses E14_E16_E18_P1_GSE195500__all_5441_local_cells.h5ad generated by source quality control.
source_h5ad = (
    ROOT
)

# Recover counts for the selected P1 cells from the QC data.
cell_ids = p1_fixed_round3.obs_names.copy()
gene_ids = ep_expression.var_names.copy()  # Stable E18.5/P1 analysis feature axis
labels = p1_fixed_round3.obs["cell_type"].reindex(cell_ids)

if labels.isna().any() or set(labels.astype(str)) != set(TYPE_ORDER):
    raise ValueError("The P1 four-type labels are incomplete.")
if not cell_ids.is_unique or not cell_ids.isin(qc_full.obs_names).all():
    raise ValueError("Selected cell IDs do not match qc_full.")
if not gene_ids.is_unique or not gene_ids.isin(qc_full.var_names).all():
    raise ValueError("Analysis genes do not match qc_full.")
if not qc_full.obs.loc[cell_ids, "qc_pass"].astype(bool).all():
    raise ValueError("Selected cells include a cell that failed the current P1 QC.")
if qc_full.var.loc[gene_ids, ["mt", "hb", "technical"]].to_numpy(dtype=bool).any():
    raise ValueError("Excluded feature classes remain in the analysis gene axis.")

X = sparse.csr_matrix(
    qc_full[cell_ids, gene_ids].layers["counts"], dtype=np.int32
)
X.sort_indices()

# Borrow gene identity and mapping fields only; do not apply the older
# all-stage QC file's P1 cell filtering to this independently reviewed set.
upstream = ad.read_h5ad(source_h5ad, backed="r")
try:
    if not upstream.var_names.equals(qc_full.var_names):
        raise ValueError("Source H5AD and qc_full have different gene axes.")
    var = upstream.var.loc[gene_ids].copy()
finally:
    upstream.file.close()

var.index.name = "feature_row_id_index"
var["gene_symbol"] = var.index.astype(str).to_numpy()
var["feature_type"] = "Gene Expression"
var["hb_feature"] = var.index.str.match(r"(?i)^Hb[ab]-")
var["hemoglobin_feature"] = var["hb_feature"]
var["integration_gene_id_candidate"] = var["canonical_gene_id"].astype(str)
var["use_for_model_input"] = True

excluded = [
    "mt_feature", "hb_feature", "ERCC_feature",
    "reporter_or_vector_feature", "technical_feature",
]
if var[excluded].to_numpy(dtype=bool).any():
    raise ValueError("A mitochondrial, hemoglobin, or technical feature remains.")
if var["eligible_for_cross_dataset_integration"].astype(bool).any():
    raise ValueError("Cross-dataset gene harmonization has not been performed.")
if not var["gene_id_type"].astype(str).eq("gene_symbol").all():
    raise ValueError("Unexpected source gene ID type.")

parent_X = sparse.csr_matrix(qc_full[:, gene_ids].layers["counts"])
var["n_cells_post_p1_qc"] = np.asarray(parent_X.getnnz(axis=0)).ravel()
var["total_counts_post_p1_qc"] = (
    np.asarray(parent_X.sum(axis=0)).ravel().astype(np.int64)
)
var["n_cells_in_export"] = np.asarray(X.getnnz(axis=0)).ravel()
var["total_counts_in_export"] = (
    np.asarray(X.sum(axis=0)).ravel().astype(np.int64)
)
var["detected_in_exported_cells"] = var["n_cells_in_export"].gt(0)

metadata_cols = [
    "original_cell_id", "barcode", "dataset_id", "library_id", "GSM",
    "sample_id", "biological_replicate_id", "stage_original",
    "stage_standardized", "technology", "cochlear_region", "genotype",
    "source_file", "author_preselected", "total_counts", "n_genes",
    "pct_hb", "pct_ribo", "neuronal_genes_detected", "blood_review",
    "non_target_review", "doublet_score", "doublet_candidate",
    "qc_pass", "qc_review_flag", "exclusion_reason",
]
metadata_cols += [
    col for col in qc_full.obs.columns
    if col.endswith("_genes_detected") and col not in metadata_cols
]
obs = qc_full.obs.loc[cell_ids, metadata_cols].copy()
obs.index.name = "cell_id_global_index"
obs["cell_id_global"] = obs.index.astype(str).to_numpy()
obs["source_cell_id"] = obs["original_cell_id"].astype(str)
obs["study_id"] = obs["dataset_id"].astype(str)
obs["stage_analysis"] = obs["stage_standardized"].astype(str)
obs["batch_id_global"] = (
    obs["dataset_id"].astype(str) + "__" + obs["library_id"].astype(str)
)
obs["integration_batch"] = obs["batch_id_global"]
obs["batch_stage_confounded"] = True

obs["total_counts_source"] = obs["total_counts"].astype(np.int64)
obs["n_features_source"] = obs["n_genes"].astype(np.int32)
nontechnical = ~qc_full.var["technical"].to_numpy(dtype=bool)
endogenous_X = sparse.csr_matrix(
    qc_full[cell_ids, nontechnical].layers["counts"]
)
obs["total_counts_endogenous"] = (
    np.asarray(endogenous_X.sum(axis=1)).ravel().astype(np.int64)
)
obs["n_genes_endogenous"] = np.asarray(
    endogenous_X.getnnz(axis=1)
).ravel()
obs["total_counts_analysis"] = (
    np.asarray(X.sum(axis=1)).ravel().astype(np.int64)
)
obs["n_genes_analysis"] = np.asarray(X.getnnz(axis=1)).ravel()

# The source has no mitochondrial or reporter features: keep these metrics NA.
obs["pct_mito"] = np.nan
obs["pct_reporter"] = np.nan
obs["technical_qc_pass"] = obs["qc_pass"].astype(bool)
obs["cell_calling_qc_pass"] = obs["author_preselected"].astype(bool)

# Blood-review signals alone are not a confirmed blood-cell identity.
# Stop if a selected review cell lacks neuronal support or has erythroid support.
blood_review = obs["blood_review"].astype(bool)
unsafe_blood_review = blood_review & (
    obs["neuronal_genes_detected"].lt(3)
    | obs["erythroid_genes_detected"].ge(2)
)
if unsafe_blood_review.any():
    raise ValueError(
        "Selected blood-review cells require inspection: "
        + ", ".join(obs.index[unsafe_blood_review].tolist())
    )
obs["blood_qc_pass"] = True
obs["blood_lineage_status"] = np.where(
    blood_review, "retained_neuronal_blood_review", "retained_non_blood"
)

# The top-5% Scrublet flag marks cells for review; it is not a doublet call.
obs["predicted_doublet"] = obs["doublet_score"].ge(1.0)
if obs["predicted_doublet"].any():
    raise ValueError("A predicted doublet is present in the selected cells.")
obs["doublet_qc_pass"] = True
obs["doublet_status"] = np.where(
    obs["doublet_candidate"].astype(bool),
    "review_candidate_retained",
    "not_predicted",
)
obs["qc_complexity_mode"] = "author_preselected_with_P1_review"
obs["qc_complexity_confidence"] = "not_quantified"
obs["qc_high_complexity_probability"] = np.nan

obs["cell_type"] = labels.astype(str).to_numpy()
obs["candidate_identity"] = obs["cell_type"]
obs["annotation_status"] = "final_stagewise_P1_SGN_subtype_annotation"
obs["selected_for_integration"] = True
obs["scanvi_label"] = "Unknown"
obs["scanvi_supervision_status"] = (
    "unlabeled_until_cross_dataset_validation"
)

initial_ids = set(p1_fixed.obs_names)
round2_ids = set(p1_fixed_round2.obs_names)
obs["candidate_source"] = np.where(
    obs.index.isin(initial_ids), "P1_initial_fixed_clusters",
    np.where(
        obs.index.isin(round2_ids),
        "P1_unresolved_cluster_1_Type_Ib",
        "P1_remaining_cluster_7_Type_Ic",
    ),
)

result = ad.AnnData(X=X, obs=obs, var=var)
result.layers["counts"] = X.copy()
result.raw = None

def ids_sha256(index):
    return hashlib.sha256(
        "\0".join(map(str, index)).encode("utf-8")
    ).hexdigest()

def matrix_sha256(matrix):
    matrix = matrix.tocsr(copy=True)
    matrix.sort_indices()
    digest = hashlib.sha256()
    digest.update(np.asarray(matrix.shape, dtype="<i8").tobytes())
    digest.update(matrix.indptr.astype("<i8").tobytes())
    digest.update(matrix.indices.astype("<i8").tobytes())
    digest.update(matrix.data.astype("<i4").tobytes())
    return digest.hexdigest()

def audit(adata):
    counts = adata.layers["counts"]
    if not sparse.isspmatrix_csr(adata.X) or not sparse.isspmatrix_csr(counts):
        raise ValueError("X and counts must both be CSR matrices.")
    if adata.X.dtype != np.int32 or counts.dtype != np.int32:
        raise ValueError("X and counts must both be int32.")
    if (adata.X != counts).nnz or (adata.X.data < 0).any():
        raise ValueError("X/counts mismatch or negative count.")
    if adata.raw is not None or any(
        len(getattr(adata, slot)) for slot in ("obsm", "obsp", "varm", "varp")
    ):
        raise ValueError("A temporary analysis result entered the export.")
    if not adata.obs_names.is_unique or not adata.var_names.is_unique:
        raise ValueError("Duplicate cell or gene IDs.")
    if len(set(map(str.casefold, adata.var_names))) != adata.n_vars:
        raise ValueError("Gene symbols are not unique after case folding.")
    if not adata.obs_names.equals(cell_ids) or not adata.var_names.equals(gene_ids):
        raise ValueError("Exported cell or gene axis changed.")
    if not np.array_equal(
        adata.obs["cell_id_global"].astype(str).to_numpy(),
        adata.obs_names.to_numpy()
    ):
        raise ValueError("cell_id_global differs from obs_names.")
    if not adata.obs["selected_for_integration"].astype(bool).all():
        raise ValueError("An exported cell is not marked as selected.")
    if not adata.obs["integration_batch"].notna().all():
        raise ValueError("Missing integration batch.")
    return {
        "ordered_cell_id_sha256": ids_sha256(adata.obs_names),
        "ordered_feature_id_sha256": ids_sha256(adata.var_names),
        "csr_counts_sha256": matrix_sha256(adata.X),
        "matrix_nnz": int(adata.X.nnz),
        "total_umi_counts": int(adata.X.sum()),
        "n_obs": int(adata.n_obs),
        "n_vars": int(adata.n_vars),
    }

fingerprints = audit(result)
type_counts = (
    result.obs["cell_type"].value_counts()
    .reindex(TYPE_ORDER, fill_value=0).to_dict()
)

result.uns["matrix_contract"] = {
    "schema_version": "1.0.0",
    "X_semantics": "raw_integer_UMI_counts",
    "counts_layer_semantics": "exact_copy_of_X",
    "sparse_format": "CSR",
    "dtype": "int32",
    "normalization_applied": False,
    "log1p_applied": False,
    "scaling_applied": False,
    "raw_present": False,
}
result.uns["selection_contract"] = {
    "schema_version": "1.0.0",
    "selection_purpose": "P1_four_SGN_types_for_later_integration",
    "source_notebook": 'fig1_P1_clustering.ipynb',
    "stable_source_object": "qc_full",
    "selected_object": "p1_fixed_round3",
    "selection_chain": [
        "P1 source counts and independent P1 QC",
        "Type II: joint P1 cluster 13",
        "Type Ia: Type I HVG2000 clusters 0, 2, 7",
        "Type Ib: Type I cluster 4 plus unresolved cluster 1",
        "Type Ic: Type I cluster 6 plus remaining cluster 7",
    ],
    "type_counts": type_counts,
    "p1_qc_parent_cells": int(qc_full.n_obs),
    "total_exported_cells": int(result.n_obs),
    "blood_review_only_retained": int(blood_review.sum()),
    "doublet_review_only_retained": int(
        obs["doublet_candidate"].astype(bool).sum()
    ),
    "annotation_status": "final_stagewise_P1_SGN_subtype_annotation",
}
result.uns["integration_contract"] = {
    "schema_version": "1.0.0",
    "count_source": "layers/counts",
    "batch_key": "integration_batch",
    "batch_definition": "GSE195500__library_id",
    "batch_stage_confounded": True,
    "scanvi_label_key": "scanvi_label",
    "scanvi_unlabeled_category": "Unknown",
    "scanvi_max_epochs": 400,
    "required_HVG_comparison_counts": np.array([2000, 3000, 5000]),
    "cross_dataset_gene_harmonization_status": "not_run",
    "cross_dataset_integration_blocker":
        "versioned_cross_dataset_gene_harmonization_not_run",
}
result.uns["source_provenance"] = {
    "schema_version": "1.0.0",
    "dataset_id": "GSE195500",
    "study_id": "GSE195500",
    "source_library_ids": ["GSE195500_L13", "GSE195500_L14"],
    "source_counts_relative_path": str(SOURCE.relative_to(ROOT)),
    "source_counts_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "source_h5ad_relative_path": str(source_h5ad.relative_to(ROOT)),
    "source_h5ad_sha256":
        hashlib.sha256(source_h5ad.read_bytes()).hexdigest(),
    "source_h5ad_role": "gene_metadata_only; not P1 cell filtering",
    "source_notebook": 'fig1_P1_clustering.ipynb',
    "source_to_export_matrix_relation":
        "selected P1 cells and stable E18.5/P1 analysis genes from qc_full raw counts",
    "p1_mitochondrial_feature_status": "absent; pct_mito not calculated",
    "export_timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
result.uns["export_contract"] = {
    "schema_version": "1.0.0",
    "output_file_name": out_path.name,
    **fingerprints,
    "excluded_feature_classes": [
        "mitochondrial", "Hba/Hbb", "ERCC", "reporter/vector", "technical"
    ],
    "saved_slots": ["X", "layers/counts", "obs", "var", "uns"],
    "normalized_matrix_saved": False,
    "HVG_result_saved": False,
    "PCA_saved": False,
    "UMAP_saved": False,
    "neighbor_graph_saved": False,
}

# Reuse an identical export and stop if the existing file differs.
if out_path.exists():
    previous = ad.read_h5ad(out_path)
    if audit(previous) != fingerprints:
        raise FileExistsError(f"Existing integration file differs: {out_path}")
    if previous.obs["cell_type"].value_counts().to_dict() != type_counts:
        raise FileExistsError(f"Existing cell-type labels differ: {out_path}")
else:
    temp_path = out_path.with_name(f"{out_path.stem}__writing_{uuid.uuid4().hex[:8]}.h5ad")
    try:
        result.write_h5ad(temp_path, compression="gzip")
        reread = ad.read_h5ad(temp_path)
        if audit(reread) != fingerprints:
            raise ValueError("Read-back fingerprints differ from the source object.")
        if reread.obs["cell_type"].value_counts().to_dict() != type_counts:
            raise ValueError("Read-back cell-type counts changed.")
        if out_path.exists():
            raise FileExistsError(f"Existing file will not be overwritten: {out_path}")
        os.replace(temp_path, out_path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


In [ ]:
#10. =========== Export P1 integration H5AD (schema 2.0)
from importlib.metadata import version

# Set the output directory path.
output_dir = ROOT
# Set the result file path; uses P1_GSE195500__L13-L14__SGN_for_integration.h5ad generated by P1 clustering export.
old_path = ROOT
# Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
new_path = ROOT

export = result.copy()
old_contracts = {key: export.uns[key] for key in (
    "matrix_contract", "selection_contract", "integration_contract",
    "source_provenance", "export_contract"
)}
old_provenance = old_contracts["source_provenance"]

# Keep the selected cells and recover their verified counts from the QC parent.
source_counts = sparse.csr_matrix(
    qc_full[export.obs_names, export.var_names].layers["counts"],
    dtype=np.int32,
)
source_counts.sort_indices()
if (source_counts != export.layers["counts"]).nnz:
    raise ValueError("Selected counts differ from the P1 QC parent.")
export.X = source_counts.copy()
export.layers["counts"] = source_counts.copy()

final_labels = p1_fixed_round3.obs["cell_type"].reindex(export.obs_names)
if final_labels.isna().any() or set(final_labels.astype(str)) != set(TYPE_ORDER):
    raise ValueError("Final P1 cell-type labels are incomplete.")

export.obs["cell_type"] = pd.Categorical(
    final_labels.astype(str), categories=TYPE_ORDER, ordered=True
)
export.obs["candidate_identity"] = final_labels.astype(str).to_numpy()
export.obs["scanvi_label"] = export.obs["cell_type"].copy()
export.obs["annotation_status"] = "final_stagewise_annotation"
export.obs["scanvi_supervision_status"] = "labeled_final_stagewise_annotation"
export.obs["selected_for_integration"] = True

# qc_full uses global cell IDs; the raw barcode remains in original_cell_id.
export.obs["source_cell_id"] = export.obs_names.astype(str).to_numpy()
export.obs["technology"] = "10x"
export.obs.drop(columns="biological_replicate_id", errors="ignore", inplace=True)

# Scrublet supplied scores and a review flag, not a validated doublet call.
for column in ("predicted_doublet", "doublet_qc_pass"):
    export.obs[column] = pd.Series(
        pd.NA, index=export.obs_names, dtype="boolean"
    )
export.obs["doublet_status"] = "scrublet_score_only"

# Preserve source IDs as candidates; cross-dataset canonical mapping is pending.
old_canonical = export.var["canonical_gene_id"].astype(str).copy()
old_mapping_status = export.var["canonical_mapping_status"].astype(str).copy()
if not old_canonical.eq(export.var["source_feature_id"].astype(str)).all():
    raise ValueError("Existing canonical IDs differ from source feature IDs.")

export.var["integration_gene_id_candidate"] = old_canonical.to_numpy()
export.var["integration_gene_id_candidate_type"] = "gene_symbol"
export.var["source_gene_id_type"] = export.var["gene_id_type"].astype(str)
export.var["source_canonical_mapping_status"] = old_mapping_status.to_numpy()
export.var["canonical_gene_id"] = ""
export.var["canonical_mapping_status"] = "not_run"
export.var["eligible_for_cross_dataset_integration"] = False
export.var["cross_dataset_feature_blocker"] = (
    "versioned_cross_dataset_gene_harmonization_not_run"
)
export.var["eligible_for_within_dataset_analysis"] = True
export.var["use_for_model_input"] = True

# The shared ribosomal gene set has not yet been fixed across datasets.
export.var["source_ribosomal_feature"] = export.var["ribosomal_feature"].astype(bool)
export.var["ribosomal_feature"] = pd.Series(
    pd.NA, index=export.var_names, dtype="boolean"
)

export.obs["total_counts_analysis"] = np.asarray(
    source_counts.sum(axis=1)
).ravel().astype(np.int64)
export.obs["n_genes_analysis"] = np.asarray(
    source_counts.getnnz(axis=1)
).ravel().astype(np.int32)
export.var["n_cells_in_export"] = np.asarray(
    source_counts.getnnz(axis=0)
).ravel().astype(np.int32)
export.var["total_counts_in_export"] = np.asarray(
    source_counts.sum(axis=0)
).ravel().astype(np.int64)
export.var["detected_in_exported_cells"] = (
    export.var["total_counts_in_export"] > 0
)

export.raw = None
for slot in ("obsm", "obsp", "varm", "varp"):
    getattr(export, slot).clear()
export.uns.clear()

type_counts = {
    name: int((export.obs["cell_type"].astype(str) == name).sum())
    for name in TYPE_ORDER
}
fingerprints = audit(export)  # Function defined in the existing #9 cell.
excluded_mask = (
    qc_full.var["mt"] | qc_full.var["hb"] | qc_full.var["technical"]
).to_numpy(dtype=bool)
excluded_ids = qc_full.var_names[excluded_mask].astype(str).tolist()

export.uns["matrix_contract"] = {
    "schema_version": "2.0.0",
    "raw_counts_available": True,
    "raw_counts_status": "available_verified",
    "counts_layer": "counts",
    "counts_layer_semantics": "raw_counts",
    "count_type": "UMI",
    "counts_verification_source": (
        "GSE195500 source count matrix; aligned to qc_full cell and gene IDs"
    ),
    "counts_dtype": "int32",
    "counts_sparse_format": "CSR",
    "X_semantics": "raw_integer_UMI_counts",
    "X_dtype": "int32",
    "X_sparse_format": "CSR",
    "X_normalization_status": "not_applied",
    "X_log1p_status": "not_applied",
    "X_scaling_status": "not_applied",
    "X_counts_relationship": "exact_copy",
    "raw_present": False,
}
export.uns["selection_contract"] = {
    "source_notebook": 'fig1_P1_clustering.ipynb',
    "source_object": "p1_fixed_round3 (labels); qc_full (counts)",
    "selection_rule": old_contracts["selection_contract"]["selection_chain"],
    "n_selected_cells": int(export.n_obs),
    "cell_type_counts": type_counts,
    "annotation_status": "final_stagewise_annotation",
    "annotation_source": "Reviewed P1 cluster and marker annotations",
    "label_policy": "all_final_cell_types",
    "blood_review_only_retained": int(
        export.obs["blood_review"].astype(bool).sum()
    ),
    "doublet_review_only_retained": int(
        export.obs["doublet_candidate"].astype(bool).sum()
    ),
}
export.uns["integration_contract"] = {
    "schema_name": "auditory_neuron_stagewise",
    "schema_version": "2.0.0",
    "input_role": "stagewise_pre_harmonization",
    "counts_layer": "counts",
    "batch_key": "integration_batch",
    "labels_key": "scanvi_label",
    "unlabeled_category": "Unknown",
    "label_policy": "all_final_cell_types",
    "gene_harmonization_status": "not_run",
    "cross_dataset_blocker": (
        "versioned_cross_dataset_gene_harmonization_not_run"
    ),
    "batch_stage_confounded": True,
    "count_model_compatible": True,
    "model_ready": False,
    "model_blockers": [
        "versioned_cross_dataset_gene_harmonization_not_run"
    ],
}
export.uns["source_provenance"] = {
    "schema_version": "2.0.0",
    "source_files": {
        "raw_counts": str(SOURCE.relative_to(ROOT)),
        "gene_metadata": str(source_h5ad.relative_to(ROOT)),
    },
    "source_file_sha256": {
        "raw_counts": old_provenance["source_counts_sha256"],
        "gene_metadata": old_provenance["source_h5ad_sha256"],
    },
    "source_notebook": 'fig1_P1_clustering.ipynb',
    "source_dataset": "GSE195500",
    "source_library": sorted(export.obs["library_id"].astype(str).unique()),
    "source_object": "qc_full",
    "source_reference_assembly": "unknown",
    "source_annotation_release": "unknown",
    "source_reference_evidence": "not_recorded",
    "ribosomal_feature_status": "pending_shared_gene_set",
    "qc_metric_definitions": {
        "total_counts_analysis": "Sum of exported counts genes per cell",
        "n_genes_analysis": "Detected exported counts genes per cell",
        "total_counts_source": "Source count matrix before feature exclusion",
        "pct_hb": "Percent of source counts assigned to Hba/Hbb genes",
        "pct_ribo": "Percent of source counts assigned to source ribosomal genes",
        "pct_mito": "Missing: mitochondrial features absent from source matrix",
        "pct_reporter": "Missing: reporter percentage not available",
        "doublet_score": "Scrublet score; candidate flag is for review only",
    },
    "migration_history": {
        "previous_filename": old_path.name,
        "previous_scanvi_policy": "all_Unknown",
        "canonical_id_action": (
            "Old source-only canonical IDs retained as candidates; "
            "canonical_gene_id cleared pending joint mapping"
        ),
        "doublet_action": (
            "Unvalidated predicted_doublet and doublet_qc_pass cleared to NA"
        ),
        "technology_action": (
            "Previous 10x_3prime_v3.1 label reduced to source-supported 10x"
        ),
        "biological_replicate_action": (
            "Library-derived biological_replicate_id removed; no animal ID verified"
        ),
        "previous_contracts": old_contracts,
    },
    "software_versions": {
        "anndata": version("anndata"),
        "scanpy": version("scanpy"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "export_date": datetime.now(timezone.utc).isoformat(),
}
export.uns["export_contract"] = {
    "output_filename": new_path.name,
    "n_obs": int(export.n_obs),
    "n_vars": int(export.n_vars),
    "saved_slots": ["X", "layers/counts", "obs", "var", "uns"],
    "excluded_feature_classes": [
        "mitochondrial", "Hba/Hbb", "ERCC", "reporter/vector", "technical"
    ],
    "excluded_feature_ids": excluded_ids,
    "ordered_cell_id_sha256": fingerprints["ordered_cell_id_sha256"],
    "ordered_feature_id_sha256": fingerprints["ordered_feature_id_sha256"],
    "X_sha256": fingerprints["csr_counts_sha256"],
    "counts_sha256": fingerprints["csr_counts_sha256"],
    "X_nnz": int(source_counts.nnz),
    "counts_nnz": int(source_counts.nnz),
    "total_counts": int(source_counts.sum()),
    "fingerprint_encoding": (
        "SHA-256; ordered IDs joined by NUL and UTF-8 encoded; "
        "sorted CSR shape/indptr/indices as little-endian int64, "
        "data as little-endian int32"
    ),
}

def verify_p1_export(adata):
    if audit(adata) != fingerprints:
        raise ValueError("Cell, gene, or count fingerprints changed.")
    labels_read = adata.obs["cell_type"].astype(str).to_numpy()
    scanvi_read = adata.obs["scanvi_label"].astype(str).to_numpy()
    expected = final_labels.astype(str).to_numpy()
    if not (
        np.array_equal(labels_read, expected)
        and np.array_equal(scanvi_read, expected)
    ):
        raise ValueError("Final cell_type or scanvi_label changed.")
    if not adata.obs["predicted_doublet"].isna().all():
        raise ValueError("Unverified doublet calls were restored.")
    if not adata.obs["doublet_qc_pass"].isna().all():
        raise ValueError("Unverified doublet QC status was restored.")
    if not adata.var["ribosomal_feature"].isna().all():
        raise ValueError("Shared ribosomal status was inferred prematurely.")
    if not adata.var["canonical_gene_id"].astype(str).eq("").all():
        raise ValueError("Unvalidated canonical IDs were restored.")
    if not np.array_equal(
        adata.var["integration_gene_id_candidate"].astype(str).to_numpy(),
        old_canonical.to_numpy(),
    ):
        raise ValueError("Source gene ID candidates changed.")
    if not np.array_equal(
        adata.obs["original_cell_id"].astype(str).to_numpy(),
        result.obs["original_cell_id"].astype(str).to_numpy(),
    ):
        raise ValueError("Original cell IDs changed.")
    if not np.array_equal(
        adata.obs["source_cell_id"].astype(str).to_numpy(),
        adata.obs_names.astype(str).to_numpy(),
    ):
        raise ValueError("source_cell_id does not match qc_full IDs.")
    if adata.uns["integration_contract"]["model_ready"]:
        raise ValueError("Gene harmonization has not been completed.")

verify_p1_export(export)

# Confirm the old path is the export being replaced.
if old_path.exists():
    previous = ad.read_h5ad(old_path)
    if audit(previous) != fingerprints:
        raise ValueError(f"Old file differs from the current P1 selection: {old_path}")
    del previous

# Set the temporary working directory or file path.
temp_dir = ROOT / f"p1_h5ad_export_{uuid.uuid4().hex}"
temp_dir.mkdir(parents=True)
temp_path = temp_dir / new_path.name
try:
    export.write_h5ad(temp_path, compression="gzip")
    reread = ad.read_h5ad(temp_path)
    verify_p1_export(reread)
    del reread

    os.replace(temp_path, new_path)
    final_check = ad.read_h5ad(new_path)
    verify_p1_export(final_check)
    del final_check

    if old_path.exists():
        old_path.unlink()
finally:
    if temp_path.exists():
        temp_path.unlink()
    temp_dir.rmdir()

print(f"Saved {new_path}: {export.n_obs} cells × {export.n_vars} genes; {type_counts}")